<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/02.%20Data%20Preprocessing/ch02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 2: Data Preprocessing

## Mengapa Preprocessing Penting?

Data mentah (*raw data*) hampir tidak pernah siap langsung dimasukkan ke model ML. Preprocessing adalah langkah kritis yang mencakup:
- **Scaling**: Menyamakan skala fitur agar tidak mendominasi satu sama lain
- **Encoding**: Mengubah data kategorik menjadi format numerik
- **Imputation**: Menangani nilai yang hilang (*missing values*)
- **Transformation**: Mengubah distribusi data

Tanpa preprocessing yang baik, banyak algoritma ML (khususnya berbasis gradient atau jarak) akan menghasilkan performa yang buruk.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_diabetes, make_classification
from sklearn.model_selection import train_test_split

# Load datasets
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

diabetes = load_diabetes()
Xd, yd = diabetes.data, diabetes.target

print(f"Iris shape: {X.shape}")
print(f"Diabetes shape: {Xd.shape}")


## 2.1 Feature Scaling

### StandardScaler (Z-score Normalization)

StandardScaler mengubah setiap fitur agar memiliki mean = 0 dan standard deviation = 1:

$$z = \frac{x - \mu}{\sigma}$$

di mana $\mu$ adalah mean dan $\sigma$ adalah standard deviation dari **training data**.

**Kapan digunakan:** Model berbasis gradient descent (SVM, Logistic Regression, Neural Networks), PCA, dan algoritma berbasis jarak.

### MinMaxScaler

Memetakan setiap fitur ke rentang $[0, 1]$ (atau rentang kustom $[a, b]$):

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

**Kapan digunakan:** Ketika distribusi tidak Gaussian, atau saat perlu menjaga nilai nol.

### RobustScaler

Menggunakan **median** dan **IQR** (Inter-Quartile Range) sehingga tahan terhadap outlier:

$$x_{\text{robust}} = \frac{x - Q_{50}}{Q_{75} - Q_{25}}$$

**Kapan digunakan:** Data dengan banyak outlier.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Apply scalers to iris sepal length (with outlier injected)
feature_idx = 0  # sepal length
X_col = X_train[:, feature_idx].copy()

# Add artificial outliers to demonstrate RobustScaler
X_col_outlier = np.append(X_col, [20.0, 25.0])  # extreme outliers

scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].hist(X_col_outlier, bins=20, color='gray', alpha=0.7)
axes[0].set_title('Original (with outliers)')
axes[0].set_xlabel('Value')

for ax, (name, scaler) in zip(axes[1:], scalers.items()):
    scaled = scaler.fit_transform(X_col_outlier.reshape(-1, 1)).flatten()
    ax.hist(scaled, bins=20, alpha=0.7, color='steelblue')
    ax.set_title(f'{name}\nmean={scaled.mean():.2f}, std={scaled.std():.2f}')
    ax.set_xlabel('Scaled Value')

plt.tight_layout()
plt.show()

# Comparison table
print("=== Scaler Comparison ===")
print(f"{'Scaler':<20} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8}")
print("-" * 56)
for name, scaler in scalers.items():
    s = scaler.fit_transform(X_col_outlier.reshape(-1, 1)).flatten()
    print(f"{name:<20} {s.min():>8.3f} {s.max():>8.3f} {s.mean():>8.3f} {s.std():>8.3f}")


## 2.2 Encoding Variabel Kategorik

### OneHotEncoder

Mengubah setiap kategori menjadi vektor biner. Untuk variabel dengan $K$ kategori, menghasilkan $K$ (atau $K-1$ jika `drop='first'`) kolom biner. Digunakan untuk variabel **nominal** (tidak ada urutan).

**Contoh:** Warna = ['Merah', 'Biru', 'Hijau'] → [1,0,0], [0,1,0], [0,0,1]

### OrdinalEncoder

Mengubah kategori yang **berurutan** (ordinal) menjadi integer sesuai urutannya.

**Contoh:** Pendidikan = ['SMA', 'S1', 'S2', 'S3'] → [0, 1, 2, 3]

### LabelEncoder

Mengubah label target menjadi integer. Khusus digunakan untuk **target variable** ($y$), bukan untuk fitur input.


In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

# Create sample categorical data
data_cat = pd.DataFrame({
    'kota': ['Jakarta', 'Bandung', 'Surabaya', 'Bandung', 'Jakarta', 'Surabaya', 'Medan'],
    'pendidikan': ['S1', 'SMA', 'S2', 'S3', 'SMA', 'S1', 'S2'],
    'gender': ['M', 'F', 'M', 'F', 'M', 'F', 'M']
})

print("Original Data:")
print(data_cat)

# OneHotEncoder for 'kota' (nominal)
ohe = OneHotEncoder(sparse_output=False, drop='first')
kota_encoded = ohe.fit_transform(data_cat[['kota']])
print(f"\nOneHotEncoder 'kota' (drop=first):")
kota_df = pd.DataFrame(kota_encoded, columns=ohe.get_feature_names_out(['kota']))
print(kota_df)

# OrdinalEncoder for 'pendidikan' (ordinal)
edu_categories = [['SMA', 'S1', 'S2', 'S3']]
oe = OrdinalEncoder(categories=edu_categories)
edu_encoded = oe.fit_transform(data_cat[['pendidikan']])
print(f"\nOrdinalEncoder 'pendidikan':")
print(dict(zip(data_cat['pendidikan'], edu_encoded.flatten())))

# LabelEncoder for target
le = LabelEncoder()
gender_encoded = le.fit_transform(data_cat['gender'])
print(f"\nLabelEncoder 'gender': {dict(zip(le.classes_, le.transform(le.classes_)))}")


## 2.3 Imputasi Missing Values

Missing values terjadi ketika data tidak tercatat untuk beberapa observasi. Strategi imputasi umum:

| Strategi | Formula | Cocok Untuk |
|---|---|---|
| **Mean** | $\hat{x} = \bar{x}$ | Data numerik, distribusi simetris |
| **Median** | $\hat{x} = \text{median}(X)$ | Data numerik dengan outlier |
| **Most Frequent** | $\hat{x} = \text{mode}(X)$ | Data kategorik |
| **KNN** | $\hat{x} = \frac{1}{k}\sum_{j \in N_k} x_j$ | Data kompleks dengan pola |
| **Constant** | $\hat{x} = c$ | Kasus khusus (misal: 0, 'Unknown') |


In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

# Create dataset with missing values
np.random.seed(42)
n_samples = 100
X_missing = np.random.randn(n_samples, 4)

# Introduce missing values randomly (15% missing)
missing_mask = np.random.rand(n_samples, 4) < 0.15
X_missing[missing_mask] = np.nan

print(f"Dataset shape: {X_missing.shape}")
print(f"Missing values per column:")
for i in range(4):
    n_missing = np.isnan(X_missing[:, i]).sum()
    print(f"  Feature {i}: {n_missing} missing ({n_missing/n_samples*100:.1f}%)")

# Strategy 1: Mean imputation
imputer_mean = SimpleImputer(strategy='mean')
X_imputed_mean = imputer_mean.fit_transform(X_missing)

# Strategy 2: Median imputation
imputer_median = SimpleImputer(strategy='median')
X_imputed_median = imputer_median.fit_transform(X_missing)

# Strategy 3: KNN imputation
imputer_knn = KNNImputer(n_neighbors=5)
X_imputed_knn = imputer_knn.fit_transform(X_missing)

print(f"\nOriginal missing count: {np.isnan(X_missing).sum()}")
print(f"After mean imputation:  {np.isnan(X_imputed_mean).sum()}")
print(f"After median imputation:{np.isnan(X_imputed_median).sum()}")
print(f"After KNN imputation:   {np.isnan(X_imputed_knn).sum()}")

# Compare imputed values for feature 0
missing_idx = np.where(np.isnan(X_missing[:, 0]))[0][:5]
print(f"\nSample comparison (Feature 0, first 5 missing indices):")
print(f"{'Index':<8} {'Mean':>8} {'Median':>8} {'KNN':>8}")
for idx in missing_idx:
    print(f"{idx:<8} {X_imputed_mean[idx,0]:>8.3f} {X_imputed_median[idx,0]:>8.3f} {X_imputed_knn[idx,0]:>8.3f}")


In [ ]:
# Visualize imputation strategies
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Mean Imputed', 'Median Imputed', 'KNN Imputed (k=5)', 'Original (no NaN)']
datasets = [X_imputed_mean, X_imputed_median, X_imputed_knn, X_missing]
colors = ['#2196F3', '#4CAF50', '#FF9800', 'gray']

for ax, data, title, color in zip(axes, datasets, titles, colors):
    if np.isnan(data).any():
        valid = data[~np.isnan(data[:, 0]), 0]
    else:
        valid = data[:, 0]
    ax.hist(valid, bins=20, color=color, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('Feature 0 Value')
    ax.set_ylabel('Count')
    ax.axvline(np.nanmean(data[:, 0]), color='red', linestyle='--', label='mean')
    ax.legend()

plt.tight_layout()
plt.show()


## 2.4 Transformasi Distribusi

Banyak algoritma ML (terutama linear models) mengasumsikan fitur terdistribusi normal. `PowerTransformer` membantu menormalkan distribusi skewed.

### Box-Cox Transformation

Hanya untuk nilai **positif** ($x > 0$):

$$y = \begin{cases} \dfrac{x^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\ \ln(x) & \text{if } \lambda = 0 \end{cases}$$

### Yeo-Johnson Transformation

Ekstensi Box-Cox untuk nilai positif **dan** negatif:

$$y = \begin{cases} \dfrac{(x+1)^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0, x \geq 0 \\ \ln(x+1) & \text{if } \lambda = 0, x \geq 0 \\ \dfrac{-((-x+1)^{2-\lambda} - 1)}{2-\lambda} & \text{if } \lambda \neq 2, x < 0 \\ -\ln(-x+1) & \text{if } \lambda = 2, x < 0 \end{cases}$$


In [ ]:
from sklearn.preprocessing import PowerTransformer, QuantileTransformer

# Create highly skewed data (log-normal distribution)
np.random.seed(42)
X_skewed = np.random.exponential(scale=2.0, size=(500, 1))

# Apply transformations
pt_boxcox = PowerTransformer(method='box-cox')   # requires positive values
pt_yeo = PowerTransformer(method='yeo-johnson')
qt = QuantileTransformer(output_distribution='normal', random_state=42)

X_boxcox = pt_boxcox.fit_transform(X_skewed)
X_yeo = pt_yeo.fit_transform(X_skewed)
X_quantile = qt.fit_transform(X_skewed)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
datasets_t = [(X_skewed, 'Original (Exponential)', 'gray'),
              (X_boxcox, 'Box-Cox Transform', '#2196F3'),
              (X_yeo, 'Yeo-Johnson Transform', '#4CAF50'),
              (X_quantile, 'Quantile Transform', '#FF9800')]

from scipy import stats
for ax, (data, title, color) in zip(axes, datasets_t):
    flat = data.flatten()
    ax.hist(flat, bins=30, density=True, color=color, alpha=0.7)
    # Overlay normal curve
    xmin, xmax = ax.get_xlim()
    xx = np.linspace(xmin, xmax, 100)
    mu, sig = flat.mean(), flat.std()
    ax.plot(xx, stats.norm.pdf(xx, mu, sig), 'k--', lw=2)
    # Shapiro-Wilk test for normality
    _, pval = stats.shapiro(flat[:200])  # limit to 200 for speed
    ax.set_title(f'{title}\nShapiro p={pval:.4f}')
    ax.set_xlabel('Value')

plt.tight_layout()
plt.show()

print(f"Box-Cox lambda: {pt_boxcox.lambdas_[0]:.4f}")
print(f"Yeo-Johnson lambda: {pt_yeo.lambdas_[0]:.4f}")


## 2.5 Polynomial Features

Untuk menangkap hubungan non-linear, kita dapat menciptakan fitur polinomial. Untuk fitur $[x_1, x_2]$ dengan `degree=2`, fitur yang dibuat adalah:

$$[1, x_1, x_2, x_1^2, x_1 x_2, x_2^2]$$

Jumlah fitur baru untuk $p$ fitur dengan derajat $d$ adalah $\binom{p+d}{d}$.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Generate non-linear 1D data
np.random.seed(42)
X_1d = np.sort(np.random.uniform(-3, 3, 80)).reshape(-1, 1)
y_1d = 0.5 * X_1d.ravel()**3 - 2 * X_1d.ravel()**2 + X_1d.ravel() + np.random.randn(80) * 2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_plot = np.linspace(-3.2, 3.2, 300).reshape(-1, 1)

for ax, degree in zip(axes, [1, 3, 5]):
    poly = PolynomialFeatures(degree=degree, include_bias=True)
    lr = LinearRegression()

    X_poly = poly.fit_transform(X_1d)
    lr.fit(X_poly, y_1d)

    y_plot = lr.predict(poly.transform(x_plot))
    r2 = lr.score(X_poly, y_1d)
    n_features = X_poly.shape[1]

    ax.scatter(X_1d, y_1d, alpha=0.5, s=20, color='gray', label='Data')
    ax.plot(x_plot, y_plot, 'r-', lw=2, label=f'degree={degree}')
    ax.set_title(f'Polynomial degree={degree}\nR²={r2:.3f}, n_features={n_features}')
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.legend()
    ax.set_ylim(-40, 20)

plt.tight_layout()
plt.show()

print("Polynomial feature names (degree=2, 2 features):")
pf2 = PolynomialFeatures(degree=2)
pf2.fit(np.zeros((1, 2)))
print(pf2.get_feature_names_out(['x1', 'x2']))


## 2.6 Complete Preprocessing Pipeline

Dalam praktik nyata, preprocessing mencakup berbagai langkah yang harus dilakukan secara benar dan terkoordinasi. `Pipeline` dan `ColumnTransformer` scikit-learn memungkinkan kita membangun preprocessing yang robust dan bebas data leakage.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Create synthetic mixed-type dataset
np.random.seed(0)
n = 300
age = np.random.randint(18, 70, n).astype(float)
income = np.random.exponential(40000, n)
score = np.random.randn(n) * 20 + 60
gender = np.random.choice(['M', 'F'], n)
city = np.random.choice(['Jakarta', 'Bandung', 'Surabaya', 'Medan'], n)

# Introduce 10% missing values in age and income
age[np.random.choice(n, 30, replace=False)] = np.nan
income[np.random.choice(n, 30, replace=False)] = np.nan

df = pd.DataFrame({'age': age, 'income': income, 'score': score,
                   'gender': gender, 'city': city})
target = (income > np.nanmedian(income)).astype(int)
target = np.where(np.isnan(income), 0, target)

X_full = df
y_full = target

X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.3, random_state=42)

# Define numeric and categorical columns
num_cols = ['age', 'income', 'score']
cat_cols = ['gender', 'city']

# Build preprocessing for numeric features
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Build preprocessing for categorical features
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False))
])

# Combine with ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

# Full ML pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
])

full_pipeline.fit(X_tr, y_tr)
y_pred_full = full_pipeline.predict(X_te)
print(f"Full Pipeline Accuracy: {accuracy_score(y_te, y_pred_full):.4f}")
print(f"Training samples: {X_tr.shape[0]}, Test samples: {X_te.shape[0]}")

# Show transformed shape
X_tr_transformed = preprocessor.fit_transform(X_tr)
print(f"\nInput features: {X_full.shape[1]} -> Transformed features: {X_tr_transformed.shape[1]}")


## Ringkasan: Key Takeaways Chapter 2

| Teknik | Kapan Digunakan | API scikit-learn |
|---|---|---|
| **StandardScaler** | Algoritma berbasis gradient, PCA, SVM | `StandardScaler()` |
| **MinMaxScaler** | Neural networks, data non-Gaussian | `MinMaxScaler()` |
| **RobustScaler** | Data dengan outlier signifikan | `RobustScaler()` |
| **OneHotEncoder** | Fitur nominal (tanpa urutan) | `OneHotEncoder()` |
| **OrdinalEncoder** | Fitur ordinal (ada urutan) | `OrdinalEncoder()` |
| **SimpleImputer** | Missing values sederhana | `SimpleImputer(strategy=...)` |
| **KNNImputer** | Missing values kompleks | `KNNImputer(n_neighbors=5)` |
| **PowerTransformer** | Normalkan distribusi skewed | `PowerTransformer(method='yeo-johnson')` |
| **PolynomialFeatures** | Relasi non-linear | `PolynomialFeatures(degree=d)` |

### Aturan Emas Preprocessing
1. **Selalu fit pada training data saja** — jangan gunakan informasi dari test set
2. **Gunakan Pipeline** — memastikan preprocessing yang konsisten antara training dan inference
3. **Urutan penting**: Imputation → Encoding → Scaling → Model
4. **Drop='first' di OneHotEncoder** — hindari multikolinieritas sempurna (dummy variable trap)
